## Prediction Using as Sample Input Data    

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd
import pickle

In [2]:
model= load_model('model.h5')

In [3]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [4]:
## Get onehotencoder, label encoder, standard scaler
# load encoders
with open('onehotencode.pkl','rb') as f:
    onehotencoder=pickle.load(f)
with open('label_encode_gender.pkl','rb') as f:
    labelencoder=pickle.load(f)
with open('scaler.pkl','rb') as f:
    scaler=pickle.load(f)

In [5]:
## Sample Data to evaluate

input_data= {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'Exited': 1
}

In [6]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,600,France,Male,40,3,60000,2,1,1,1


In [7]:
one_hot_geo= onehotencoder.transform([input_df['Geography']])
one_hot_geo.toarray()
one_hot_geo_df=pd.DataFrame(one_hot_geo.toarray(),columns=onehotencoder.get_feature_names_out())
one_hot_geo_df

c:\Users\ssan\Downloads\Cursor_Projects\cursor\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [8]:
input_df['Gender']=labelencoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,600,France,1,40,3,60000,2,1,1,1


In [9]:
input_df= pd.concat([input_df.drop('Geography',axis=1),one_hot_geo_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,1,1.0,0.0,0.0


In [10]:
scaled_data= scaler.transform(input_df)
scaled_data

array([[-0.52289309,  0.91025899,  0.10571138, -0.69166803, -0.26876324,
         0.80937733,  0.63968351,  0.96922337,  1.97154467,  0.99277609,
        -0.57677292, -0.57234647]])

In [11]:
# Predicted model

prediction=model.predict(scaled_data)
prediction

1/1 [==============================] - 0s 132ms/step


array([[92080.984]], dtype=float32)

In [12]:
prediction_proba=prediction[0][0]
prediction_proba

92080.984

In [13]:
# if prediction_proba > 0.5:
#     print("Customer likely to churn")
# else:
#     print("Customer not likely to churn")